# R00_common — Shared Infrastructure

Defines all data loaders, evaluation protocol, statistical helpers, and quantum kernel builders used by R01–R0N notebooks.  
Load via `%run recheck/notebooks/R00_common.ipynb` from any other notebook.

In [1]:

# ── Cell 1: Imports & Constants ─────────────────────────────────────────────
import os, sys, warnings, hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedGroupKFold, StratifiedShuffleSplit
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score,
    brier_score_loss, log_loss, confusion_matrix
)
from scipy import stats

warnings.filterwarnings("ignore")

# ── Constants ────────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# Majority-class baseline accuracies (populated from data audit in R03)
BASE_MAJORITY = {
    'pima':   0.651,
    'sylhet': 0.615,
    'cdc':    0.861,
    '130hosp': 0.888,
}

# Root of the project (two levels up from this notebook)
PROJECT_ROOT = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path('.').resolve()
# When run via %run the cwd is the project root
DATA_ROOT = PROJECT_ROOT / 'data'

print("Imports OK")
print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"DATA_ROOT     : {DATA_ROOT}")
print(f"SEED={SEED}  BASE_MAJORITY={BASE_MAJORITY}")


Imports OK
PROJECT_ROOT  : C:\Research\Adaptive-Quantum-Kernel-Selection\recheck\notebooks
DATA_ROOT     : C:\Research\Adaptive-Quantum-Kernel-Selection\recheck\notebooks\data
SEED=42  BASE_MAJORITY={'pima': 0.651, 'sylhet': 0.615, 'cdc': 0.861, '130hosp': 0.888}


In [2]:

# ── Cell 2: Data Loaders ─────────────────────────────────────────────────────

def load_pima():
    """Load Pima Indians Diabetes dataset.
    
    Returns
    -------
    X : pd.DataFrame  (768, 8)
    y : np.ndarray    (768,) int  — 0/1
    """
    path = DATA_ROOT / 'pima' / 'pima-indians-diabetes.csv'
    col_names = [
        'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
    ]
    df = pd.read_csv(path, header=None, names=col_names)
    # If the file already has a header row that matches, drop it
    if df.iloc[0]['Pregnancies'] == 'Pregnancies':
        df = df.iloc[1:].reset_index(drop=True)
    # Cast
    df = df.apply(pd.to_numeric, errors='coerce')
    X = df.drop(columns=['Outcome'])
    y = df['Outcome'].astype(int).values
    return X, y


def load_sylhet():
    """Load Sylhet early-stage diabetes dataset (UCI 529).

    Returns
    -------
    X         : pd.DataFrame  (520, 16) — encoded
    y         : np.ndarray    (520,) int
    group_ids : np.ndarray    (520,) — unique profile hashes for GroupKFold
    """
    path = DATA_ROOT / 'sylhet' / 'early_stage_diabetes_uci529.csv'
    df = pd.read_csv(path)

    # Target
    target_col = 'class'
    y = (df[target_col].str.strip().str.lower() == 'positive').astype(int).values

    X = df.drop(columns=[target_col]).copy()

    # Encode categoricals
    for col in X.select_dtypes(include='object').columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

    # Build group_ids: hash of each row's feature values (mimics participant profile)
    def _row_hash(row):
        h = hashlib.md5(str(row.values.tolist()).encode()).hexdigest()
        return h
    group_ids = np.array([_row_hash(row) for _, row in X.iterrows()])

    return X, y, group_ids


def load_cdc(n=2000, seed=42):
    """Load CDC Diabetes Health Indicators dataset (UCI 891).

    Performs a stratified subsample of `n` rows.

    Returns
    -------
    X                : pd.DataFrame  (n, n_features)
    y                : np.ndarray    (n,) int
    subsample_indices: np.ndarray    (n,) — original row indices
    """
    path = DATA_ROOT / 'cdc' / 'cdc_diabetes_health_indicators_uci891.csv'
    df = pd.read_csv(path)

    target_col = 'Diabetes_binary'
    # Drop ID if present
    drop_cols = [c for c in ['ID'] if c in df.columns]
    df = df.drop(columns=drop_cols)

    y_full = df[target_col].astype(int).values
    X_full = df.drop(columns=[target_col])

    # Stratified subsample
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    sub_idx, _ = next(sss.split(X_full, y_full))

    X = X_full.iloc[sub_idx].reset_index(drop=True)
    y = y_full[sub_idx]
    return X, y, sub_idx


def load_130hosp(n=2000, seed=42):
    """Load Diabetes 130-Hospitals dataset.

    Target: readmitted == '<30'
    Encodes categoricals with LabelEncoder, drops high-cardinality ID cols.
    Stratified subsample of `n` rows.

    Returns
    -------
    X : pd.DataFrame  (n, n_features)
    y : np.ndarray    (n,) int
    """
    path = DATA_ROOT / 'diabetes_130_hospitals' / 'diabetic_data.csv'
    df = pd.read_csv(path, low_memory=False)

    # Target
    y_full = (df['readmitted'] == '<30').astype(int).values

    # Drop ID and target columns
    drop_cols = ['encounter_id', 'patient_nbr', 'readmitted']
    # Also drop cols with >50% missing / '?' values
    X_full = df.drop(columns=drop_cols).copy()

    # Replace '?' with NaN
    X_full = X_full.replace('?', np.nan)

    # Encode object columns (label-encode; NaN -> -1 placeholder)
    for col in X_full.select_dtypes(include='object').columns:
        X_full[col] = X_full[col].fillna('__missing__')
        le = LabelEncoder()
        X_full[col] = le.fit_transform(X_full[col].astype(str))

    # Cast everything to float (NaN stays NaN for numeric cols)
    X_full = X_full.apply(pd.to_numeric, errors='coerce')

    # Stratified subsample
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    sub_idx, _ = next(sss.split(X_full, y_full))

    X = X_full.iloc[sub_idx].reset_index(drop=True)
    y = y_full[sub_idx]
    return X, y


print("Data loaders defined: load_pima, load_sylhet, load_cdc, load_130hosp")


Data loaders defined: load_pima, load_sylhet, load_cdc, load_130hosp


In [3]:

# ── Cell 3: Metric Helpers ────────────────────────────────────────────────────

def gmean_score(y_true, y_pred):
    """Geometric mean of sensitivity and specificity."""
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape != (2, 2):
        raise ValueError("gmean_score requires binary classification.")
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sensitivity * specificity))


def _compute_fold_metrics(y_true, y_pred, y_prob):
    """Compute all per-fold metrics."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan
    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan

    return {
        'accuracy':          accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision':         precision_score(y_true, y_pred, zero_division=0),
        'recall':            recall_score(y_true, y_pred, zero_division=0),
        'specificity':       specificity,
        'f1':                f1_score(y_true, y_pred, zero_division=0),
        'gmean':             gmean_score(y_true, y_pred),
        'mcc':               matthews_corrcoef(y_true, y_pred),
        'roc_auc':           roc_auc,
        'pr_auc':            pr_auc,
        'brier':             brier_score_loss(y_true, y_prob),
        'log_loss':          log_loss(y_true, y_prob, eps=1e-7),
        'n_pos_test':        int(y_true.sum()),
        'n_neg_test':        int((1 - y_true).sum()),
    }


print("Metric helpers defined: gmean_score, _compute_fold_metrics")


Metric helpers defined: gmean_score, _compute_fold_metrics


In [4]:

# ── Cell 4: evaluate_model ────────────────────────────────────────────────────

def evaluate_model(
    clf, X, y,
    cv_strategy='repeated_stratified',
    n_repeats=10, n_splits=5,
    groups=None,
    random_state=42
):
    """
    Evaluate classifier using cross-validation with in-fold preprocessing.

    Parameters
    ----------
    clf           : sklearn-compatible classifier (unfitted)
    X             : pd.DataFrame or np.ndarray
    y             : np.ndarray (binary 0/1)
    cv_strategy   : 'repeated_stratified' | 'stratified_group'
    n_repeats     : number of CV repeats
    n_splits      : folds per repeat
    groups        : group labels for StratifiedGroupKFold (Sylhet)
    random_state  : RNG seed

    Returns
    -------
    metrics_df         : pd.DataFrame — one row per fold
    oof_preds          : dict with keys 'y_true', 'y_pred', 'y_prob' (concatenated)
    majority_baseline_acc : float
    """
    import copy
    from sklearn.base import clone

    X_arr = X.values if hasattr(X, 'values') else np.array(X)
    y_arr = np.array(y, dtype=int)

    majority_baseline_acc = max(y_arr.mean(), 1 - y_arr.mean())

    fold_records = []
    all_y_true, all_y_pred, all_y_prob = [], [], []

    if cv_strategy == 'stratified_group' and groups is not None:
        # Manual n_repeats for StratifiedGroupKFold
        sgkf = StratifiedGroupKFold(n_splits=n_splits)
        fold_splits = []
        for rep in range(n_repeats):
            rng = np.random.RandomState(random_state + rep)
            perm = rng.permutation(len(y_arr))
            X_perm, y_perm = X_arr[perm], y_arr[perm]
            g_perm = groups[perm] if groups is not None else None
            for train_idx, test_idx in sgkf.split(X_perm, y_perm, groups=g_perm):
                fold_splits.append((
                    perm[train_idx], perm[test_idx], rep
                ))
    else:
        # RepeatedStratifiedKFold
        rskf = RepeatedStratifiedKFold(
            n_splits=n_splits, n_repeats=n_repeats, random_state=random_state
        )
        fold_splits = [
            (train_idx, test_idx, rep)
            for rep, (train_idx, test_idx) in enumerate(
                [(t, v) for t, v in rskf.split(X_arr, y_arr)]
            )
        ]
        # Reconstruct proper rep indices
        fold_splits = []
        for fold_i, (train_idx, test_idx) in enumerate(rskf.split(X_arr, y_arr)):
            rep_i = fold_i // n_splits
            fold_splits.append((train_idx, test_idx, rep_i))

    for fold_i, (train_idx, test_idx, rep_i) in enumerate(fold_splits):
        X_tr, X_te = X_arr[train_idx], X_arr[test_idx]
        y_tr, y_te = y_arr[train_idx], y_arr[test_idx]

        # ── In-fold preprocessing ──────────────────────────────────────────
        # 1) Impute on train, apply to test
        imputer = SimpleImputer(strategy='mean')
        X_tr = imputer.fit_transform(X_tr)
        X_te = imputer.transform(X_te)

        # 2) Scale on train, apply to test
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)

        # ── Fit & predict ──────────────────────────────────────────────────
        clf_clone = clone(clf)
        clf_clone.fit(X_tr, y_tr)
        y_pred = clf_clone.predict(X_te)

        if hasattr(clf_clone, 'predict_proba'):
            y_prob = clf_clone.predict_proba(X_te)[:, 1]
        elif hasattr(clf_clone, 'decision_function'):
            df_raw = clf_clone.decision_function(X_te)
            # Sigmoid to get pseudo-probabilities
            y_prob = 1.0 / (1.0 + np.exp(-df_raw))
        else:
            y_prob = y_pred.astype(float)

        # ── Metrics ───────────────────────────────────────────────────────
        fold_metrics = _compute_fold_metrics(y_te, y_pred, y_prob)
        fold_metrics['fold']   = fold_i
        fold_metrics['repeat'] = rep_i
        fold_records.append(fold_metrics)

        all_y_true.extend(y_te.tolist())
        all_y_pred.extend(y_pred.tolist())
        all_y_prob.extend(y_prob.tolist())

    metrics_df = pd.DataFrame(fold_records)
    oof_preds = {
        'y_true': np.array(all_y_true),
        'y_pred': np.array(all_y_pred),
        'y_prob': np.array(all_y_prob),
    }
    return metrics_df, oof_preds, majority_baseline_acc


print("evaluate_model defined")


evaluate_model defined


In [5]:
def make_angle_kernel(n_qubits):
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_qubits), rotation='Y')
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation='Y')
        return qml.probs(wires=range(n_qubits))
    return lambda x1, x2: float(circuit(x1, x2)[0])

def make_zz_kernel(n_qubits):
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x1, x2):
        # ZZ feature map with Hadamard + Rz + entangling CNOT + Rz + CNOT
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            qml.RZ(2.0 * x1[i], wires=i)
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                qml.CNOT(wires=[i, j])
                qml.RZ(2.0 * (np.pi - x1[i]) * (np.pi - x1[j]), wires=j)
                qml.CNOT(wires=[i, j])
        # Adjoint
        for i in reversed(range(n_qubits)):
            for j in reversed(range(i + 1, n_qubits)):
                qml.CNOT(wires=[i, j])
                qml.RZ(-2.0 * (np.pi - x2[i]) * (np.pi - x2[j]), wires=j)
                qml.CNOT(wires=[i, j])
        for i in reversed(range(n_qubits)):
            qml.RZ(-2.0 * x2[i], wires=i)
            qml.Hadamard(wires=i)
        return qml.probs(wires=range(n_qubits))
    return lambda x1, x2: float(circuit(x1, x2)[0])

def make_z_kernel(n_qubits):
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x1, x2):
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            qml.RZ(2.0 * x1[i], wires=i)
        for i in reversed(range(n_qubits)):
            qml.RZ(-2.0 * x2[i], wires=i)
            qml.Hadamard(wires=i)
        return qml.probs(wires=range(n_qubits))
    return lambda x1, x2: float(circuit(x1, x2)[0])

def compute_gram_matrix(kernel_func, X1, X2=None):
    """
    Computes Gram matrix between X1 and X2 (or X1 and X1 if X2 is None).
    Optimized for symmetric computation when X2 is None.
    """
    X1 = np.asarray(X1)
    n1 = len(X1)
    if X2 is None:
        K = np.ones((n1, n1), dtype=np.float64)
        for i in range(n1):
            for j in range(i + 1, n1):
                val = float(kernel_func(X1[i], X1[j]))
                K[i, j] = val
                K[j, i] = val
        return K
    else:
        X2 = np.asarray(X2)
        n2 = len(X2)
        K = np.zeros((n1, n2), dtype=np.float64)
        for i in range(n1):
            for j in range(n2):
                K[i, j] = float(kernel_func(X1[i], X2[j]))
        return K

print('PennyLane Quantum Kernel factories defined: Angle, ZZ, Z, and Gram matrix compute.')

PennyLane Quantum Kernel factories defined: Angle, ZZ, Z, and Gram matrix compute.


In [6]:
def compute_classification_metrics(y_true, y_pred, y_prob=None):
    """
    Computes a comprehensive dictionary of binary classification metrics.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    auc = roc_auc_score(y_true, y_prob) if (y_prob is not None and len(np.unique(y_true)) > 1) else np.nan
    pr_auc = average_precision_score(y_true, y_prob) if (y_prob is not None and len(np.unique(y_true)) > 1) else np.nan
    brier = brier_score_loss(y_true, y_prob) if y_prob is not None else np.nan
    
    return {
        'accuracy': float(acc),
        'balanced_accuracy': float(bal_acc),
        'precision': float(prec),
        'recall': float(rec),
        'specificity': float(spec),
        'f1': float(f1),
        'roc_auc': float(auc),
        'pr_auc': float(pr_auc),
        'brier': float(brier),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp)
    }

def nadeau_bengio_ttest(diffs, n_train, n_test):
    """
    Corrected resampled t-test for repeated cross-validation (Nadeau & Bengio 2003).
    diffs: array of paired metric differences across all runs and folds (e.g., 50 differences for 10x5-fold).
    """
    diffs = np.asarray(diffs)
    k_r = len(diffs)
    mean_diff = np.mean(diffs)
    var_diff = np.var(diffs, ddof=1)
    
    correction = (1.0 / k_r) + (n_test / n_train)
    corrected_var = var_diff * correction
    
    if corrected_var <= 0:
        return 0.0, 1.0
        
    t_stat = mean_diff / np.sqrt(corrected_var)
    p_val = 2.0 * (1.0 - stats.t.cdf(np.abs(t_stat), df=k_r - 1))
    return float(t_stat), float(p_val)

def holm_bonferroni(p_values):
    """
    Holm-Bonferroni correction for multiple hypothesis testing.
    """
    p_values = np.asarray(p_values)
    n = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    
    adjusted = np.empty(n)
    running_max = 0.0
    for i, p in enumerate(sorted_p):
        adj = p * (n - i)
        running_max = max(running_max, adj)
        adjusted[i] = min(running_max, 1.0)
        
    out = np.empty(n)
    out[order] = adjusted
    return out

def bootstrap_metric_ci(y_true, y_pred, metric_func=accuracy_score, n_boot=1000, alpha=0.05, seed=42):
    """
    Computes empirical bootstrap confidence intervals.
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    boot_stats = []
    
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        val = metric_func(y_true[idx], y_pred[idx])
        boot_stats.append(val)
        
    boot_stats = np.array(boot_stats)
    low = np.percentile(boot_stats, 100.0 * (alpha / 2.0))
    high = np.percentile(boot_stats, 100.0 * (1.0 - alpha / 2.0))
    point = metric_func(y_true, y_pred)
    return float(point), float(low), float(high)

print('Statistical testing tools defined: Nadeau-Bengio corrected t-test, Holm-Bonferroni, Bootstrap CI.')

Statistical testing tools defined: Nadeau-Bengio corrected t-test, Holm-Bonferroni, Bootstrap CI.


In [7]:
import pennylane as qml

def make_angle_kernel(X_train, X_test, scale=1.0, n_wires=None):
    n_qubits = min(X_train.shape[1], 8) if n_wires is None else n_wires
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x):
        for i in range(n_qubits):
            qml.RX(scale * x[i], wires=i)
        return qml.state()
    V_tr = np.array([circuit(x[:n_qubits]) for x in X_train])
    V_te = np.array([circuit(x[:n_qubits]) for x in X_test])
    K_tr = np.abs(V_tr @ V_tr.conj().T) ** 2
    K_te = np.abs(V_te @ V_tr.conj().T) ** 2
    return K_tr, K_te

def make_zz_kernel(X_train, X_test, scale=1.0, reps=1):
    n_qubits = min(X_train.shape[1], 8)
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x):
        for r in range(reps):
            for i in range(n_qubits):
                qml.Hadamard(wires=i)
                qml.RZ(2.0 * scale * x[i], wires=i)
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])
                qml.RZ(2.0 * scale * (np.pi - x[i]) * (np.pi - x[i+1]), wires=i + 1)
                qml.CNOT(wires=[i, i + 1])
        return qml.state()
    V_tr = np.array([circuit(x[:n_qubits]) for x in X_train])
    V_te = np.array([circuit(x[:n_qubits]) for x in X_test])
    K_tr = np.abs(V_tr @ V_tr.conj().T) ** 2
    K_te = np.abs(V_te @ V_tr.conj().T) ** 2
    return K_tr, K_te

def make_z_kernel(X_train, X_test, scale=1.0):
    n_qubits = min(X_train.shape[1], 8)
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x):
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            qml.RZ(scale * x[i], wires=i)
        return qml.state()
    V_tr = np.array([circuit(x[:n_qubits]) for x in X_train])
    V_te = np.array([circuit(x[:n_qubits]) for x in X_test])
    K_tr = np.abs(V_tr @ V_tr.conj().T) ** 2
    K_te = np.abs(V_te @ V_tr.conj().T) ** 2
    return K_tr, K_te

def make_amplitude_kernel(X_train, X_test):
    norm_tr = np.linalg.norm(X_train, axis=1, keepdims=True)
    norm_tr[norm_tr == 0] = 1.0
    X_tr_norm = X_train / norm_tr
    norm_te = np.linalg.norm(X_test, axis=1, keepdims=True)
    norm_te[norm_te == 0] = 1.0
    X_te_norm = X_test / norm_te
    K_tr = np.clip(np.abs(X_tr_norm @ X_tr_norm.T) ** 2, 0.0, 1.0)
    K_te = np.clip(np.abs(X_te_norm @ X_tr_norm.T) ** 2, 0.0, 1.0)
    return K_tr, K_te, norm_tr.flatten(), norm_te.flatten()
